<a href="https://colab.research.google.com/github/ssykes-eth/ETH_273-0003-00L/blob/project/Project_4/deepfake_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 4:  Deepfake Generation 🍎 ↔️ 🍊

Welcome to the final and most exciting project of the course! Over the next few cells, you will build a
**generative AI pipeline** capable of swapping fruits in images, replacing apples with oranges and
vice versa, while perfectly preserving the background scene!

This is not just a fun visual trick. The techniques you will use here  (diffusion models, latent
spaces, cross-attention conditioning, and adversarial discrimination) are the foundations
behind tools like Stable Diffusion and DALL·E. By the end of this notebook you will have built your
own version from scratch. Let's go! 🚀

```
                        LATENT DIFFUSION DEEPFAKE PIPELINE

1. Raw Input Data (Pixel Space)

[🍊 Source Crop]    [🍎 Target Composite]    [🍎 Target Redacted]    [🎭 Target Mask]
  (fruit to           (full scene)             (background only)       (fruit location)
   generate)
       │                    │                        │                       │
       ▼                    ▼                        ▼                       ▼

2. Encoding & Feature Extraction

 (Fruit Encoder)      (VAE Encoder)            (VAE Encoder)           (Downsample)
  CNN → 128-dim     128×128 → 16×16×4        128×128 → 16×16×4       128×128 → 16×16
       │                    │                        │                       │
       ▼                    ▼                        ▼                       ▼
[Fruit Embedding]   [Latent Composite]       [Latent Redacted]        [Latent Mask]


3. DDPM Reverse Process (T=1000 → t=0)

                    ┌─── Conditioning ───┐
                    │                    │
             [Time Embedding]    [Fruit Embedding]
                    │                    │
                    └────────┬───────────┘
                             │
                        cond = t_emb + fruit_emb
                             │
                             │  (enters via cross-attention)
                             ▼
  [x_T ~ N(0,I)] ──►  ┌─────────────────┐ ◄── [Redacted + Mask]
   (pure noise)       │                 │      (spatial concat)
                      │  U-Net predicts │
                      │  noise pattern  │
                      └────────┬────────┘
                               │
                               ▼
                      [Reverse Process]
                               │
                               ▼
                            [x_{t-1}]
                               │
                               └──── (repeat T times) ────┐
                                                          │
                                                          ▼

4. Decoding (Latent → Pixel Space)

                                              [Clean latent z_0]
                                                     │
                                                     ▼
                                               (VAE Decoder)
                                              16×16×4 → 128×128×3
                                                     │
                                                     ▼
                                              [🖼️ Deepfake Image]
                                          (orange replaced by apple)
```

In [ ]:
#@title ⚙️  Setup. This will take a few minutes (click ▶ to run)

import os

DATASET_DIR = 'dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

# Files hosted on polybox
FILES = {
    'fruits_train.pt': 'https://polybox.ethz.ch/index.php/s/PXL8XkNdQtTxDb8/download',
    'fruits_test.pt':  'https://polybox.ethz.ch/index.php/s/HbdTnppx6t5btyB/download',
}

for fname, url in FILES.items():
    path = f'{DATASET_DIR}/{fname}'
    if os.path.exists(path):
        print(f"{fname}: already present, skipping download")
        continue
    print(f"Downloading {fname}...")
    ret = os.system(f'curl -L -C - --fail -o "{path}" "{url}"')
    if ret != 0 or not os.path.exists(path):
        raise RuntimeError(f"Failed to download {fname} from {url}")

for f in ['fruits_train.pt', 'fruits_test.pt']:
    size = os.path.getsize(f'{DATASET_DIR}/{f}') / (1024**2)
    print(f"{f}: {size:.0f} MB")

In [ ]:
#@title 📦 Imports & Configuration (click ▶ to run)
!pip install -q diffusers accelerate

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from diffusers import AutoencoderKL
import numpy as np
import matplotlib.pyplot as plt
import random
import os

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Step 1: Loading and Freezing the VAE ⬇️ ❄️

Working directly in pixel space (128×128 images with 3 colour channels) is very expensive.
The solution used by modern systems like Stable Diffusion is elegant: **compress the image into a much smaller latent space first**, and run all the diffusion there.

We use a **Variational Autoencoder (VAE)** for this compression which has already been trained on millions of images.

The VAE has two parts:
- **Encoder**: takes a pixel image `(3, 128, 128)` and compresses it to a latent tensor `(4, 16, 16)`.
  That is a **48× reduction** in size
- **Decoder**: takes the latent `(4, 16, 16)` and reconstructs the pixel image `(3, 128, 128)`.


In [ ]:
vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse",
    torch_dtype=torch.float32
).to(device)

# 🎯 TO-DO: Set the vae to evaluation mode
#vae. ...

# 🎯 TO-DO: Iterate through the VAE parameters (vae.parameters())
# and set param.requires_grad to False
# for param in ...:
#     ...

# Test the VAE: encode then decode an image
dummy = torch.randn(1, 3, 128, 128).to(device)
with torch.no_grad():
    latent = vae.encode(dummy).latent_dist.sample()
    reconstructed = vae.decode(latent).sample

print(f"Input:   {dummy.shape}")           # (1, 3, 128, 128)
print(f"Latent:  {latent.shape}")          # (1, 4, 16, 16)
print(f"Decoded: {reconstructed.shape}")   # (1, 3, 128, 128)
print(f"Compression: {128*128*3} pixels → {16*16*4} latents ({128*128*3 / (16*16*4):.0f}× reduction)")
print("VAE loaded ✅")

## Step 2:  Encoding the Entire Dataset into Latent Space  🖼️ ➡️ 🌌

Now that we have our frozen VAE encoder, we use it to **pre-encode the full dataset once** and save
the result.

For every image in the dataset we encode four versions:
- **Composite**:  the full scene with the fruit in place
- **Redacted**:  the background with the fruit removed
- **Crop**: the fruit alone
- **Mask**: position of the fruit

One important detail: the VAE expects pixel values in the range `[-1, 1]`, but our images are stored
in `[0, 1]`. The `to_vae_range` function handles this rescaling before encoding.

After encoding, the latent tensors are multiplied by the **VAE scaling factor `0.18215`**. This is a
standard normalisation constant used by the Stable Diffusion VAE to bring the latent values into a
range that is well-suited for the noise schedule of the diffusion process.

In [ ]:
def encode_dataset_to_latents(data_path, vae, device, batch_size=16):
    data = torch.load(data_path, weights_only=False)

    # Convert float16 to float32 for VAE
    composites = data['composites'].float()
    masks = data['masks'].float()
    redacted = data['redacted'].float()
    crops = data['crops'].float()
    labels = data['labels']

    print(f"Encoding {len(labels)} images to latent space...")

    def to_vae_range(x):
        return x * 2.0 - 1.0

    latent_composites = []
    latent_redacted = []
    latent_crops = []
    downsampled_masks = []

    with torch.no_grad():
        for i in range(0, len(labels), batch_size):
            batch_comp = to_vae_range(composites[i:i+batch_size]).to(device)
            batch_red = to_vae_range(redacted[i:i+batch_size]).to(device)
            batch_crop = to_vae_range(crops[i:i+batch_size]).to(device)
            batch_mask = masks[i:i+batch_size]

            z_comp = vae.encode(batch_comp).latent_dist.sample() * 0.18215
            # 🎯 TO-DO: we also need to encode our target images (`batch_red`)
            # and our cropped images (`batch_crop`)
            # z_red = ...
            # z_crop = ...

            mask_down = F.interpolate(batch_mask, size=(16, 16), mode='nearest')

            latent_composites.append(z_comp.cpu())
            latent_redacted.append(z_red.cpu())
            latent_crops.append(z_crop.cpu())
            downsampled_masks.append(mask_down.cpu())

            if (i // batch_size) % 20 == 0:
                print(f"  {i}/{len(labels)}")

    # Free the pixel-space data
    del composites, masks, redacted, crops
    import gc; gc.collect()

    latent_data = {
        'composites': torch.cat(latent_composites),
        'redacted': torch.cat(latent_redacted),
        'crops': torch.cat(latent_crops),
        'masks': torch.cat(downsampled_masks),
        'labels': labels,
    }

    print(f"Composites latent shape: {latent_data['composites'].shape}")
    print("Done ✅")
    return latent_data

latent_train = encode_dataset_to_latents(f'{DATASET_DIR}/fruits_train.pt', vae, device)
latent_test = encode_dataset_to_latents(f'{DATASET_DIR}/fruits_test.pt', vae, device)

print("Latent datasets loaded ✅")

## 📂 Step 3:  Dataset & DataLoader

The `LatentFruitDataset` class is a standard PyTorch `Dataset` that loads our pre-encoded latent
tensors from disk and serves them one sample at a time.

For each sample it returns a dictionary containing five tensors:

| Key | Shape | Description |
|-----|-------|-------------|
| `composite` | `(4, 16, 16)` | The full scene in latent space |
| `redacted` | `(4, 16, 16)` | Background only (fruit removed) |
| `crop` | `(4, 16, 16)` | The fruit only |
| `mask` | `(1, 16, 16)` | Binary mask showing where the fruit is |
| `label` | scalar | `0` = apple, `1` = orange |

The `DataLoader` wraps the dataset and handles **batching**. With `batch_size=64`, every forward pass processes 64
images at once.

In [ ]:
class LatentFruitDataset(Dataset):
    def __init__(self, data_dict):
        self.composites = data_dict['composites']
        self.masks = data_dict['masks']
        self.redacted = data_dict['redacted']
        self.crops = data_dict['crops']
        self.labels = data_dict['labels']

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # 🎯 TO-DO: Return a dictionary containing the 'composite', 'mask',
        # 'redacted', 'crop', and 'label' for the given index `idx`
        # return {
        #     ...
        # }
        pass #remove this line after implementing

train_dataset = LatentFruitDataset(latent_train)
test_dataset = LatentFruitDataset(latent_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, pin_memory=True)

batch = next(iter(train_loader))
print(f"Latent composite: {batch['composite'].shape}")   # (64, 4, 16, 16)
print(f"Latent mask:      {batch['mask'].shape}")        # (64, 1, 16, 16)
print(f"Latent redacted:  {batch['redacted'].shape}")    # (64, 4, 16, 16)
print(f"Latent crop:      {batch['crop'].shape}")        # (64, 4, 16, 16)
print("Latent DataLoader ready ✅")

In [ ]:
#@title Data Visualization (click ▶ to run)

from torchvision.transforms.functional import resize
from torchvision.transforms import InterpolationMode
DISPLAY_SIZE = 512

def fix_black_ring(composite, mask, redacted):
    """Fill black ring pixels with background color."""
    # Pixels that are black (near zero) but not in the background
    black_pixels = (composite.sum(dim=0, keepdim=True) < 0.05) & (mask > 0.5)
    # Replace with background
    fixed = composite.clone()
    fixed = torch.where(black_pixels.expand_as(fixed), redacted, fixed)
    return fixed

@torch.no_grad()
def visualize_data_sample(data_path, vae, device, n=6):
    data = torch.load(data_path, weights_only=False)
    apple_idx = (data['labels'] == 0).nonzero(as_tuple=True)[0]
    orange_idx = (data['labels'] == 1).nonzero(as_tuple=True)[0]
    n_each = n // 2
    selected = torch.cat([
        apple_idx[torch.randperm(len(apple_idx))[:n_each]],
        orange_idx[torch.randperm(len(orange_idx))[:n_each]],
    ]).tolist()

    fig, axes = plt.subplots(4, n, figsize=(3*n, 10))
    row_labels = ['Composite\n(full image)', 'Mask\n(fruit location)',
                  'Redacted\n(background only)', 'Crop\n(fruit only)']

    for col, idx in enumerate(selected):
        # Fix black ring first, then resize
        comp_fixed = fix_black_ring(data['composites'][idx].float(),
                                    data['masks'][idx].float(),
                                    data['redacted'][idx].float())

        comp = resize(comp_fixed.unsqueeze(0),
                      [DISPLAY_SIZE, DISPLAY_SIZE],
                      interpolation=InterpolationMode.BICUBIC)[0]
        red = resize(data['redacted'][idx].float().unsqueeze(0),
                     [DISPLAY_SIZE, DISPLAY_SIZE],
                     interpolation=InterpolationMode.BICUBIC)[0]
        crop_img = resize(data['crops'][idx].float().unsqueeze(0),
                          [DISPLAY_SIZE, DISPLAY_SIZE],
                          interpolation=InterpolationMode.BICUBIC)[0]
        mask = resize(data['masks'][idx].float().unsqueeze(0),
                      [DISPLAY_SIZE, DISPLAY_SIZE],
                      interpolation=InterpolationMode.NEAREST)[0]

        label = 'apple' if data['labels'][idx] == 0 else 'orange'
        axes[0, col].imshow(comp.permute(1,2,0).clamp(0,1).numpy())
        axes[0, col].set_title(label, fontsize=11)
        axes[1, col].imshow(mask[0].numpy(), cmap='gray')
        axes[2, col].imshow(red.permute(1,2,0).clamp(0,1).numpy())
        axes[3, col].imshow(crop_img.permute(1,2,0).clamp(0,1).numpy())
        for row in range(4):
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])

    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=11, rotation=0, labelpad=80, va='center')

    plt.suptitle('Dataset', fontsize=14)
    plt.tight_layout()
    plt.show()
    del data
    import gc; gc.collect()

visualize_data_sample(f'{DATASET_DIR}/fruits_train.pt', vae, device)

## Step 4:  Fruit Encoder

The `LatentFruitEncoder` is a small convolutional network that takes the **latent crop** of a fruit
(shape `(B, 4, 16, 16)`) and produces a compact **embedding vector** of shape `(B, 128)`.

This embedding is the conditioning signal for the diffusion model. It answers the question:
*"which fruit should be generated?"* The embedding is injected into the U-Net at every level via
cross-attention, guiding the denoising process toward producing the correct fruit.

In [ ]:
class LatentFruitEncoder(nn.Module):
    """
    Encode a fruit's latent representation into a conditioning embedding.
    Input: VAE latent (4, 16, 16) instead of pixel image (3, 128, 128).
    """
    def __init__(self, in_channels=4, embed_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, 2, 1),
            nn.GroupNorm(8, 64),
            nn.GELU(),

            # 🎯🎯 TO-DO:
            # Let's add the second block
            # 1. A second convolutional layer (`nn.Conv2d`) that takes 64 input channels and outputs 128. (Hint: use kernel_size=3, stride=2, padding=1)
            # 2. A Group Normalization layer (`nn.GroupNorm`) with 8 groups for those 128 channels.
            # 3. A GELU activation function (`nn.GELU()`).
            # nn.Conv2d(...),
            # nn.GroupNorm(...),
            # nn.GELU(),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),

            # 🎯 TO-DO:
            # We need to map our flattened features to the desired embedding dimension.
            # Add a linear layer (`nn.Linear`) that takes 128 input features and outputs `embed_dim`.
            # nn.Linear(...),
        )

    def forward(self, x):
        return self.net(x)

## 🧱 Step 5:  Building Blocks of the U-Net

---

### `ConvBlock` Double Convolution with Normalisation

The `ConvBlock` is the main component of the U-Net. It applies the same sequence **twice**:
The **3×3 convolution** slides a small filter across the feature map, learning local spatial
patterns. Applying it twice (rather than once with a larger kernel) gives the block a wider
effective receptive field while using fewer parameters.

**GroupNorm** normalises the activations within each group of channels. Without normalisation, the training becomes unstable and slow.

**GELU** is the non-linear activation function. Without non-linearities, stacking many layers would
be mathematically equivalent to just one linear transformation and the network could not learn
complex patterns.

---

### `SinusoidalPosEmb`  Timestep Embedding

Tells the U-Net where in the denoising process it currently is.

---

### `CrossAttention`  Conditioning the U-Net on the Fruit Embedding

This is how the fruit embedding actually influences what the U-Net
generates.

The U-Net's intermediate feature map is reshaped into a sequence of
spatial tokens — each pixel position becomes one token. Then:

$$Q = W_Q \cdot X_{\text{features}} \quad K = W_K \cdot \text{cond} \quad V = W_V \cdot \text{cond}$$

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d}}\right) V$$

- **Q (Query)** comes from the U-Net features: each spatial position asks *"what should I look like?"*
- **K (Key)** and **V (Value)** come from the fruit embedding: they answer *"given this fruit, here
  is what you should look like."*

The softmax score determines how strongly each spatial position attends to the conditioning signal.
Pixels in the centre of the fruit region attend strongly while background pixels attend weakly and mostly
pass through the residual connection unchanged.

In [ ]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        emb = torch.log(torch.tensor(10000.0, device=t.device)) / (half - 1)
        emb = torch.exp(torch.arange(half, device=t.device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, 1, 1),
            nn.GroupNorm(8, out_ch),
            nn.GELU(),

            # 🎯 TO-DO:
            # For this second convolution, both the input AND output
            #    should have the same number of channels (`out_ch`)
            # (Use kernel_size=3, stride=1, padding=1)
            # 2. Add another `nn.GroupNorm` (with 8 groups for `out_ch` channels).
            # 3. Add a final `nn.GELU()` activation function.
            # nn.Conv2d(...),
            # nn.GroupNorm(...),
            # nn.GELU(),
        )

    def forward(self, x):
        # 🎯 TO-DO:
        # return ...
        pass #remove this line after implementing

class CrossAttention(nn.Module):
    def __init__(self, feature_dim, embed_dim=128, num_heads=4, num_tokens=4):
        super().__init__()
        # 🎯 TO-DO:
        #self.num_heads = ...
        # 🎯 TO-DO:
        #self.num_tokens = ...

        #HINT: Calculate `self.head_dim` by dividing `feature_dim` by `num_heads`.
        #(use integer division // )
        # 🎯 TO-DO:
        #self.head_dim = ...
        self.scale = self.head_dim ** -0.5

        # Keys (K) and Values (V) from the condition embedding
        # 1. A Linear layer (`nn.Linear`) mapping from `embed_dim` to `feature_dim * num_tokens`.
        # 2. A GELU activation function (`nn.GELU()`).
        # 3. Another Linear layer mapping from `feature_dim * num_tokens` to `feature_dim * num_tokens * 2`.
        # (We multiply by 2 at the end because this single layer will smoothly output BOTH our Keys and Values at once!)
        # 🎯 TO-DO:
        # self.to_kv_tokens = nn.Sequential(
        #     ...
        # )

        # Queries (Q) from the actual image features
        # `nn.Linear` layer called `self.to_q` that maps `feature_dim` to `feature_dim`.
        # 🎯 TO-DO:
        # self.to_q = ...

        self.to_out = nn.Sequential(nn.Linear(feature_dim, feature_dim), nn.GELU())
        self.norm_features = nn.LayerNorm(feature_dim)
        self.norm_out = nn.LayerNorm(feature_dim)

    def forward(self, x, cond):
        B, C, H, W = x.shape
        residual = x

        x_flat = x.permute(0, 2, 3, 1).reshape(B, H * W, C)
        x_flat = self.norm_features(x_flat)

        # 🎯 TO-DO: HINT pass (`x_flat`) through the `self.to_q` layer
        # q = ...

        kv = self.to_kv_tokens(cond)
        kv = kv.reshape(B, self.num_tokens, 2, C)

        # Split the Keys and Values
        # The `kv` tensor currently holds both our Keys and Values
        # extract `k` by taking index 0 of the last dimension, and `v` by taking index 1.
        # (HINT: You can use standard slicing, like `kv[:, :, 0]` and `kv[:, :, 1]`)
        # 🎯 TO-DO:
        # k, v = ...

        q = q.reshape(B, H * W, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        k = k.reshape(B, self.num_tokens, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        v = v.reshape(B, self.num_tokens, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        # 1. Use `torch.matmul()` to multiply `q` with `k`.
        # # HINT: You must transpose the last two dimensions of `k` first! Use `k.transpose(-2, -1)`.
        # 2. Multiply that entire result by our stabilizing factor, `self.scale`.
        # 🎯 TO-DO:
        # attn = ...

        # Apply the softmax function to your `attn` tensor along the very last dimension (`dim=-1`).
        # 🎯 TO-DO:
        # attn = ...

        # use `torch.matmul()` to multiply your softmaxed `attn` probabilities with your values `v`.
        # 🎯 TO-DO:
        # out = ...

        out = out.permute(0, 2, 1, 3).reshape(B, H * W, C)
        out = self.to_out(out)
        out = self.norm_out(out)
        out = out.reshape(B, H, W, C).permute(0, 3, 1, 2)

        return residual + out

## 🏗️ Step 6: The Latent U-Net

The `LatentUNet` is a U-Net that operates entirely in latent space. Its input is a concatenation of
three latent tensors along the channel dimension:

$$x_{\text{input}} = \text{cat}(z_{\text{noisy}},\ z_{\text{redacted}},\ \text{mask}) \quad \rightarrow \quad (B,\ 9,\ 16,\ 16)$$

> 💡 **Tip for implementation:** We encourage you to build your `enc1`, `enc2`, the bottleneck, `dec2`, and `dec1` blocks **with** the `CrossAttention` mechanism. These 2 layers along with fine tuning will be enough to get satisfying results for this project.

> ✨ **Worth Trying:** After you have successfully finished the entire project, you can always come back here to experiment! If you want to push things further, try adding an entirely new level to both your encoder and decoder.

In [ ]:
class LatentUNet(nn.Module):
    """
    U-Net operating in VAE latent space.
    Input: 16×16 latents. Only 2 downsampling levels: 16→8→4

    Spatial input: noisy_latent (4ch) + redacted_latent (4ch) + mask (1ch) = 9 channels
    Output: predicted noise (4 channels, matching latent dim)
    """
    def __init__(self, in_channels=9, out_channels=4,
                 base_ch=128, embed_dim=128):
        super().__init__()
        ch = base_ch

        # Time + condition embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(ch),
            nn.Linear(ch, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
        )

        # 🎯 TO-DO: Build the Encoder
        # 1. `enc1`: ConvBlock mapping `in_channels` to `ch`.
        # 2. `enc2`: ConvBlock mapping `ch` to `ch * 2`.
        # 3. `down`: Use nn.MaxPool2d(2) to halve the spatial dimensions.
        # self.enc1 = ...
        # self.enc2 = ...
        # self.down = ...

        # 🎯 TO-DO: Build the Bottleneck
        # `bottleneck`: ConvBlock mapping `ch * 2` to `ch * 2`.
        # self.bottleneck = ...

        # 🎯 TO-DO: Build the Decoder
        # 1. `up2` & `up1`: Use nn.ConvTranspose2d(in, out, kernel=2, stride=2).
        # 2. `dec2`: ConvBlock mapping `ch * 4` (concatenated) to `ch`.
        # 3. `dec1`: ConvBlock mapping `ch * 2` (concatenated) to `ch`.
        # self.up2 = ...
        # self.dec2 = ...
        # self.up1 = ...
        # self.dec1 = ...

        # 🎯 TO-DO: Final Output Layer
        # `out_conv`: Use nn.Conv2d with a kernel size of 1 to map `ch` to `out_channels`.
        # self.out_conv = ...

        # 🎯 TO-DO: Add Cross-Attention
        # Instantiate `CrossAttention` for each level, passing the correct `feature_dim`.
        # (HINT: check the output channels of the blocks you just created above!)
        # self.attn_enc1 = ...
        # self.attn_enc2 = ...
        # self.attn_bottleneck = ...
        # self.attn_dec2 = ...
        # self.attn_dec1 = ...

    def forward(self, x_noisy, t, redacted, mask, fruit_embed):
        # Combine time and fruit conditioning
        t_emb = self.time_mlp(t)

        # 🎯 TO-DO: add the time embedding (`t_emb`) and the fruit embedding (`fruit_embed`)
        # cond = ...

        # 🎯 TO-DO: Prepare the spatial input!
        # Use `torch.cat()` to stack `x_noisy`, `redacted`, and `mask`
        # x = ...([..., ..., ...], dim=1)

        # 🎯 TO-DO: The Encoder Journey
        # 1. Pass `x` through `self.enc1`, then apply `self.attn_enc1` (remember to pass `cond` to the attention!).
        # 2. Downsample `e1` using `self.down`, pass that through `self.enc2`, and apply `self.attn_enc2`.
        # e1 = ...
        # e1 = ...
        # e2 = ...
        # e2 = ...

        # 🎯 TO-DO: The Bottleneck
        # Downsample `e2` using `self.down`, pass it through `self.bottleneck`, and apply `self.attn_bottleneck`.
        # b = ...
        # b = ...

        # 🎯 TO-DO: The Decoder Journey
        # Here we use skip connections to bring back high-resolution details!
        # 1. Pass the result through `self.dec2` and apply `self.attn_dec2`.
        # 2. Upsample `d2` using `self.up1`. Concatenate that with `e1` (along `dim=1`).
        # 3. Pass the result through `self.dec1` and apply `self.attn_dec1`.
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        # d2 = ...
        # d1 = ...
        # d1 = ...

        # 🎯 TO-DO:
        return self.out_conv(...)

## Step 7:  DDPM Noise Scheduler 🌫️

The `DDPMScheduler` adds noise during
training and removes it during inference.

### Forward process

Given a clean latent $x_0$ and a timestep $t$, we can corrupt it directly in one step:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)$$

where $\bar{\alpha}_t = \prod_{s=1}^{t}(1 - \beta_s)$ is the cumulative product of the noise
schedule. At $t=0$ the image is clean; at $t=T$ it is almost pure noise.

The $\beta_t$ values follow a **linear schedule** that grows from `0.0001` to `0.02` — small steps
early on, slightly larger steps later.

### Reverse process

At each inference step the U-Net predicts the noise $\varepsilon_\theta$, and we
compute the slightly less noisy image:

$$\mu_t = \frac{1}{\sqrt{\alpha_t}}\!\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\,\varepsilon_\theta\right)$$

$$x_{t-1} = \mu_t + \sqrt{\beta_t}\, z, \qquad z \sim \mathcal{N}(0, I) \text{ if } t > 1 \text{, else } z = 0$$


In [ ]:
class DDPMScheduler:
    def __init__(self, num_timesteps=1000, beta_start=1e-4, beta_end=0.02, device='cuda'):
        self.num_timesteps = num_timesteps
        self.device = device

        betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)

        self.betas = betas
        self.alphas = alphas
        self.alphas_cumprod = alphas_cumprod
        self.sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

    def add_noise(self, x_0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_0)
        sqrt_alpha = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_one_minus = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]

        # 🎯 TO-DO: Return the noisy image
        return ..., noise

    @torch.no_grad()
    def sample(self, model, fruit_encoder, crop, redacted, mask,
               return_intermediates=False, intermediate_steps=10):
        shape = (crop.shape[0], 4, 16, 16)  # latent shape
        x = torch.randn(shape, device=self.device)
        fruit_embed = fruit_encoder(crop)

        intermediates = []
        save_every = max(1, self.num_timesteps // intermediate_steps)

        for t_val in reversed(range(self.num_timesteps)):
            B = x.shape[0]
            t = torch.full((B,), t_val, device=self.device, dtype=torch.long)

            # 🎯 TO-DO: Predict the noise!
            # ask our UNet (`model`) to guess the noise for this specific timestep
            # pred_noise = ...

            alpha_t = self.alphas[t_val]
            alpha_bar_t = self.alphas_cumprod[t_val]
            beta_t = self.betas[t_val]

            coeff = (1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)
            mean = (1 / torch.sqrt(alpha_t)) * (x - coeff * pred_noise)

            # 🎯 TO-DO: Take a step back in time
            # we add a tiny bit of random noise back in during every step except the last one
            # 1. Write an `if` statement to check if `t_val > 0`.
            #    If true, update `x`
            # 2. Write an `else` statement (for when `t_val == 0`). Set `x` equal to the `mean`.
            # if ...:
            #     x = ...
            # else:
            #     x = ...

            if return_intermediates and (t_val % save_every == 0 or t_val == 0):
                intermediates.append(x.cpu().clone())

        if return_intermediates:
            return x, intermediates
        return x

## Step 8: Training the Diffusion Model 🏋🏻‍♀️

Here everything comes together!

### Joint training

The optimiser should include **both** the U-Net and the fruit encoder:

```python
optimizer = optim.Adam(
    list(unet.parameters()) + list(fruit_encoder.parameters()), lr=lr
)
```

### The `base_ch` parameter

`base_ch` controls the **width** of the U-Net, how many feature channels each convolutional block
produces. All other channel counts in the architecture are multiples of this value (`base_ch`,
`base_ch×2`, etc.). A larger `base_ch` means more parameters, more expressive power, but also more
memory and longer training time.

We recommend starting with `base_ch=320`, following the channel counts used in the original Stable

> 💡 We recommend starting your training process for **300 epochs**!

In [ ]:
LATENT_IMG_SIZE = 16
BATCH_SIZE = 64
NUM_EPOCHS = 300
LEARNING_RATE = 5e-5
EMBED_DIM = 128
NUM_TIMESTEPS = 1000
BASE_CH = 320
CLASS_NAMES = {0: 'apple', 1: 'orange'}

unet = LatentUNet(base_ch=BASE_CH, embed_dim=EMBED_DIM).to(device)
fruit_encoder = LatentFruitEncoder(embed_dim=EMBED_DIM).to(device)
scheduler = DDPMScheduler(num_timesteps=NUM_TIMESTEPS, device=device)

print(f"U-Net params:    {sum(p.numel() for p in unet.parameters()):,}")
print(f"Encoder params:  {sum(p.numel() for p in fruit_encoder.parameters()):,}")


def train_latent_diffusion(unet, fruit_encoder, scheduler, train_loader,
                           device, num_epochs=NUM_EPOCHS, lr=LEARNING_RATE):
    optimizer = optim.Adam(
        list(unet.parameters()) + list(fruit_encoder.parameters()), lr=lr
    )
    scheduler_lr = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    history = []

    for epoch in range(1, num_epochs + 1):
        unet.train()
        fruit_encoder.train()
        epoch_loss = 0
        n_batches = 0

        for batch in train_loader:
            z_comp = batch['composite'].to(device)     # (B, 4, 16, 16)
            z_mask = batch['mask'].to(device)          # (B, 1, 16, 16)
            z_redacted = batch['redacted'].to(device)  # (B, 4, 16, 16)
            z_crop = batch['crop'].to(device)          # (B, 4, 16, 16)
            B = z_comp.size(0)

            # Welcome to the heart of the training loop!
            # 🎯 TO-DO: Encode the Condition
            # We need to give our UNet some guidance! Pass your cropped latent (`z_crop`)
            # through the `fruit_encoder` to get your `fruit_embed`.
            # fruit_embed = ...

            # 🎯 TO-DO: Time and Noise
            # 1. Sample random timesteps `t`. Use `torch.randint(0, scheduler.num_timesteps, (B,), device=device)`.
            # 2. Generate random `noise` that has the exact same shape as your composite latents (`z_comp`).
            #    (HINT: use `torch.randn_like`)
            # t = ...
            # noise = ...

            # 🎯 TO-DO: Forward Diffusion
            # Use your `scheduler.add_noise()` to add the `noise` to your `z_comp` at timestep `t`
            # `add_noise` returns 2 things, and we just want the first element, the noisy latents
            # z_noisy, _ = ...

            # 🎯 TO-DO: Predict the Noise
            # Pass in all your ingredients to unet: `z_noisy`, `t`, `z_redacted`, `z_mask`, and `fruit_embed`.
            # pred_noise = ...

            # 🎯 TO-DO: Calculate the Loss
            # HINT: Use `F.mse_loss()`
            # loss = ...


            # 🎯 TO-DO (zero_grad)
            # optimizer...

            # 🎯 TO-DO (backward)
            # loss...

            # 3.We are clipping the gradients here to prevent them from exploding!
            torch.nn.utils.clip_grad_norm_(
                list(unet.parameters()) + list(fruit_encoder.parameters()),
                max_norm=1.0
            )

            # 🎯 TO-DO (step)
            # optimizer...

            epoch_loss += loss.item()
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        history.append(avg_loss)
        scheduler_lr.step()

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:>3}/{num_epochs} | Loss: {avg_loss:.4f}")

    return history

print(f"U-Net params: {sum(p.numel() for p in unet.parameters()):,}")

history = train_latent_diffusion(unet, fruit_encoder, scheduler, train_loader, device)

## Step 9:  Decoding Latents Back to Pixels 🖼️ ➡️ 🌌

The `decode_latents` function uses the **decoder half** of the pretrained VAE to convert our
generated latent tensors back into full-resolution pixel images.

In [ ]:
@torch.no_grad()
def decode_latents(vae, latents):
    """Decode VAE latents back to pixel images."""
    latents = latents / 0.18215  # undo the scaling
    images = vae.decode(latents.to(vae.device)).sample
    images = (images + 1.0) / 2.0  # [-1,1] → [0,1]
    return images.clamp(0, 1)

## 👀 Step 10: Visualising Reconstructions

In [ ]:
@torch.no_grad()
def show_reconstructions_latent(unet, fruit_encoder, scheduler, vae,
                                test_loader, device, n=6):
    unet.eval()
    fruit_encoder.eval()

    batch = next(iter(test_loader))
    z_crop = batch['crop'][:n].to(device)
    z_redacted = batch['redacted'][:n].to(device)
    z_mask = batch['mask'][:n].to(device)
    z_comp = batch['composite'][:n].to(device)
    labels = batch['label'][:n]

    # 🎯 TO-DO: generate new images in the latent space
    # HINT: `scheduler.sample()`
    # z_result = ...

    # Bring them to the pixel world!
    # 🎯 TO-DO: Decode the original composite latents (`z_comp`)
    # orig_images = ...
    # 🎯 TO-DO: Decode your newly generated latents (`z_result`)
    # gen_images = ...

    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
    fig.suptitle('Top: Original  |  Bottom: Generated (latent diffusion)', fontsize=13)
    for i in range(n):
        axes[0, i].imshow(orig_images[i].cpu().permute(1, 2, 0).numpy())
        axes[0, i].set_title(CLASS_NAMES[labels[i].item()], fontsize=10)
        axes[0, i].axis('off')
        axes[1, i].imshow(gen_images[i].cpu().permute(1, 2, 0).numpy())
        axes[1, i].axis('off')
    plt.tight_layout()
    plt.show()

show_reconstructions_latent(unet, fruit_encoder, scheduler, vae, test_loader, device)

In [ ]:
#@title ## 🎬 Step 11: Visualising the Denoising Process (click ▶ to run)

batch = next(iter(test_loader))
z_crop     = batch['crop'][:1].to(device)
z_redacted = batch['redacted'][:1].to(device)
z_mask     = batch['mask'][:1].to(device)
z_comp     = batch['composite'][:1].to(device)

unet.eval(); fruit_encoder.eval()
with torch.no_grad():
    z_final, intermediates = scheduler.sample(
        unet, fruit_encoder, z_crop, z_redacted, z_mask,
        return_intermediates=True, intermediate_steps=10
    )

# Decode all intermediates to pixel space and display
fig, axes = plt.subplots(1, len(intermediates), figsize=(3 * len(intermediates), 3))
fig.suptitle('Denoising process — left: pure noise   →   right: final output', fontsize=12)
for i, z_step in enumerate(intermediates):
    img = decode_latents(vae, z_step)[0].cpu().permute(1, 2, 0).numpy()
    axes[i].imshow(img.clip(0, 1))
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## Step 12:  Deepfake Generation! 🍎 ↔️ 🍊

This is the moment you have been building toward!

The `deepfake_latent` function performs the actual fruit swap. It takes images from the test set
and using the opposite fruit's embedding as the conditioning signal generates a new version of
each scene where the original fruit has been replaced.

Fill in the To-Dos, run the cell and admire your deepfakes 🎉

In [ ]:
# Visualise the step-by-step denoising on one test sample
def find_samples_per_class(loader, n_per_class=3):
    collected = {0: [], 1: []}
    for batch in loader:
        for i in range(len(batch['label'])):
            cls = batch['label'][i].item()
            if cls in collected and len(collected[cls]) < n_per_class:
                collected[cls].append({
                    'composite': batch['composite'][i],
                    'mask': batch['mask'][i],
                    'redacted': batch['redacted'][i],
                    'crop': batch['crop'][i],
                })
        if all(len(v) >= n_per_class for v in collected.values()):
            break
    return collected

@torch.no_grad()
def deepfake_latent(unet, fruit_encoder, scheduler, vae, test_loader, device, n=6):
    unet.eval()
    fruit_encoder.eval()

    samples = find_samples_per_class(test_loader, n_per_class=n//2)
    n_each = min(n // 2, len(samples[0]), len(samples[1]))

    # Target backgrounds from apples, then oranges
    target_comps = torch.stack([s['composite'] for s in samples[0][:n_each]] +
                                [s['composite'] for s in samples[1][:n_each]]).to(device)
    target_masks = torch.stack([s['mask'] for s in samples[0][:n_each]] +
                                [s['mask'] for s in samples[1][:n_each]]).to(device)
    target_redacted = torch.stack([s['redacted'] for s in samples[0][:n_each]] +
                                  [s['redacted'] for s in samples[1][:n_each]]).to(device)
    target_labels = [0] * n_each + [1] * n_each

    # Source crops: oranges then apples (swapped)
    source_crops = torch.stack([s['crop'] for s in samples[1][:n_each]] +
                                [s['crop'] for s in samples[0][:n_each]]).to(device)
    source_labels = [1] * n_each + [0] * n_each

    # 💡 Quick note on how we swap the labels! 🔄🍎➡️🍊
    # Notice how we prepared the data just above? We deliberately mixed things up!
    # We are taking the target backgrounds from apples and pairing them with the source crops
    # of oranges (and vice versa). This "swap" is exactly how we instruct our model to create a deepfake!

    # 🎯 TO-DO: generate the deepfakes!
    # z_result = ...

    # 🎯 TO-DO: decode `target_comps`
    # orig_images = ...

    # 🎯 TO-DO: decode `z_result`
    # gen_images = ...

    # 🎯 TO-DO: decode `source_crops`
    # source_images = ...

    # 🎯 TO-DO: decode `target_redacted`
    # redacted_images = ...

    actual_n = len(target_labels)
    fig, axes = plt.subplots(4, actual_n, figsize=(3 * actual_n, 11))
    row_labels = ['Original', 'Redacted\n(background)', 'Source fruit\n(conditioning)', 'DEEPFAKE']

    for i in range(actual_n):
        axes[0, i].imshow(orig_images[i].cpu().permute(1, 2, 0).numpy())
        axes[0, i].set_title(CLASS_NAMES[target_labels[i]], fontsize=10, color='green')

        axes[1, i].imshow(redacted_images[i].cpu().permute(1, 2, 0).numpy())

        axes[2, i].imshow(source_images[i].cpu().permute(1, 2, 0).numpy())
        axes[2, i].set_title(CLASS_NAMES[source_labels[i]], fontsize=10, color='blue')

        axes[3, i].imshow(gen_images[i].cpu().permute(1, 2, 0).numpy())
        axes[3, i].set_title(
            f'{CLASS_NAMES[target_labels[i]]}→{CLASS_NAMES[source_labels[i]]}',
            fontsize=10, color='red'
        )

        for row in range(4):
            axes[row, i].axis('off')

    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=11, rotation=0, labelpad=80, va='center')

    plt.suptitle('Latent Diffusion Deepfakes', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

deepfake_latent(unet, fruit_encoder, scheduler, vae, test_loader, device)

## Saving Checkpoints 💾

Last but definitely not least, let's save our hard work! It's always best practice to save your model checkpoints (the weights of your UNet and Encoder) so you don't lose your progress.

A massive congratulations on making it this far and completing the pipeline! 🎉

In [ ]:
def save_checkpoint(unet, fruit_encoder, history, filename='diffusion_checkpoint.pt'):
    torch.save({
        'unet_state': unet.state_dict(),
        'encoder_state': fruit_encoder.state_dict(),
        'history': history,

    }, filename)
    print(f"Checkpoint saved to {filename}")

save_checkpoint(unet, fruit_encoder, history)